# FPGA Weight Export — from trained checkpoint to Verilog `$readmemh` hex

Bridges the trained model to the FPGA RTL: loads `snn_fatigue_final.pth`, folds BatchNorm
into conv1, quantizes to Q6.10 (16-bit) and int8, and writes the `.hex` files the Verilog
loads. This wraps the two torch-free scripts shipped in the handoff package
(`01_model/read_pth.py`, `01_model/prep_weights.py`).

Run this locally next to the handoff package, or upload `snn_fatigue_final.pth` + the two
scripts to Colab first.

## 1. Locate the package files

In [ ]:
import os, sys
# EDIT this to point at the handoff package root (the folder with 01_model/, 02_weights_hex/, ...)
PKG = os.environ.get("HANDOFF_PKG", ".")
MODEL = os.path.join(PKG, "01_model")
assert os.path.exists(os.path.join(MODEL,"snn_fatigue_final.pth")), "set PKG to the handoff package root"
sys.path.insert(0, MODEL)
print("using:", MODEL)

## 2. Inspect the checkpoint (no PyTorch required)
`read_pth.py` parses the `.pth` without importing torch — useful for a quick sanity check of
shapes and the learned β / threshold values.

In [ ]:
os.system(f'python "{os.path.join(MODEL,"read_pth.py")}"')
print("see _model_report.txt for the full per-tensor dump")

## 3. Prepare & export weights (BN-fold + quantize + hex)
`prep_weights.py` folds BatchNorm into conv1, quantizes to Q6.10 and int8, and writes the hex
files. The line counts must be 320 / 135168 / 512 (conv1_w / fc1_w / fc2_w) — the RTL depends
on this.

In [ ]:
os.system(f'python "{os.path.join(MODEL,"prep_weights.py")}"')
# verify hex line counts
import glob
for f in sorted(glob.glob(os.path.join(PKG,"02_weights_hex","*_q610.hex"))):
    n=sum(1 for _ in open(f)); print(os.path.basename(f), n, "lines")

## 4. Next: verify the RTL uses these weights
Back in the handoff package, run `python run_all_tests.py` (with the oss-cad-suite toolchain
active) — it confirms the Verilog is bit-exact against a Python reference using exactly these
hex weights. Expected: `SUMMARY: 4/4 checks OK`.